In [0]:
# Setup - run this every time
storage_account_name = "stkalshianalytics"
storage_account_key = dbutils.secrets.get("kalshi-scope", "adls-storage-key")

spark.conf.set(
    f"fs.azure.account.key.{storage_account_name}.dfs.core.windows.net",
    storage_account_key
)

bronze_path = f"abfss://bronze@{storage_account_name}.dfs.core.windows.net"
silver_path = f"abfss://silver@{storage_account_name}.dfs.core.windows.net"
gold_path = f"abfss://gold@{storage_account_name}.dfs.core.windows.net"

print("Ready")

In [0]:
from pyspark.sql.functions import col, current_timestamp
from datetime import datetime, timezone

# Dynamically find latest bronze partition
year_folders = sorted([f.name.rstrip('/') for f in dbutils.fs.ls(f"{bronze_path}/")])
latest_year = year_folders[-1]

month_folders = sorted([f.name.rstrip('/') for f in dbutils.fs.ls(f"{bronze_path}/{latest_year}/")])
latest_month = month_folders[-1]

day_folders = sorted([f.name.rstrip('/') for f in dbutils.fs.ls(f"{bronze_path}/{latest_year}/{latest_month}/")])
latest_day = day_folders[-1]

bronze_folder = f"{bronze_path}/{latest_year}/{latest_month}/{latest_day}/"
single_file = f"{bronze_path}/{latest_year}/{latest_month}/{latest_day}/page_0000.json"

print(f"Using bronze date: {latest_year}/{latest_month}/{latest_day}")

# Read with fixed schema from first page
schema = spark.read.option("multiline", "true").json(single_file).schema
df_raw = spark.read.option("multiline", "true").schema(schema).json(bronze_folder)

df_silver = df_raw.select(
    col("ticker"),
    col("title"),
    col("event_ticker"),
    col("status"),
    col("market_type"),
    col("strike_type"),
    col("no_sub_title"),
    col("yes_sub_title"),
    col("yes_ask_dollars").cast("double").alias("yes_ask"),
    col("yes_bid_dollars").cast("double").alias("yes_bid"),
    col("no_ask_dollars").cast("double").alias("no_ask"),
    col("no_bid_dollars").cast("double").alias("no_bid"),
    col("last_price_dollars").cast("double").alias("last_price"),
    col("liquidity_dollars").cast("double").alias("liquidity"),
    col("volume_24h_fp").cast("double").alias("volume_24h"),
    col("volume_fp").cast("double").alias("volume_total"),
    col("open_interest_fp").cast("double").alias("open_interest"),
    col("close_time").cast("timestamp"),
    col("open_time").cast("timestamp"),
    col("expected_expiration_time").cast("timestamp"),
    col("is_provisional"),
    col("mve_collection_ticker"),
    col("result"),
    col("can_close_early"),
    current_timestamp().alias("silver_updated_at")
).filter(
    (col("market_type") == "binary") &
    (col("mve_collection_ticker").isNull())
)

print(f"Silver row count: {df_silver.count()}")

In [0]:
(df_silver.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(f"{silver_path}/kalshi_markets")
)

print("Silver Delta table written successfully")
df_verify = spark.read.format("delta").load(f"{silver_path}/kalshi_markets")
print(f"Rows in silver table: {df_verify.count()}")